In [1]:
import os
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types
from typing import Optional,Dict,Any

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.CRITICAL)

print("Libraries imported")

Libraries imported


In [2]:
# 1. 加载同目录下的 .env 文件
load_dotenv()

# 2. 从系统环境变量中读取模型名称（如果没读到，默认用 deepseek-chat）
# 注意：LiteLlm 在底层会自动去寻找 os.environ["DEEPSEEK_API_KEY"]，所以我们甚至不需要手动赋给它！
MODEL_NAME = os.getenv("DEEPSEEK_MODEL", "deepseek/deepseek-chat")
llm = LiteLlm(model=MODEL_NAME)

print(llm.llm_client.completion(model=llm.model,
                                messages=[{"role": "user", "content": "你好，请问你准备好开始了吗？"}],
                                tools=[]))
print("🤖 DeepSeek 回复：")

print("\nDeepSeek is ready for use.")

ModelResponse(id='3e6cfc43-c433-4476-8fee-f633c2f8ef47', created=1788097953, model='deepseek-v4-flash', object='chat.completion', system_fingerprint='a26a7955944dc5c60445bff77fac9c8e', choices=[Choices(finish_reason='stop', index=0, message=Message(content='你好！我已经准备好了！😊\n\n无论你是想聊天、问问题、寻求帮助，还是需要处理文档、分析数据、编程辅助，我都在这里随时待命。请告诉我你需要什么，我会尽我所能为你提供帮助！\n\n现在，请开始吧——你遇到了什么问题，或者有什么想聊的吗？', role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None), provider_specific_fields={})], usage=Usage(completion_tokens=66, prompt_tokens=12, total_tokens=78, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=12))
🤖 DeepSeek 回复：

DeepSeek is ready for use.


In [3]:
## 设置AgentCaller

In [4]:
from helper import make_agent_caller

print("Libraries imported")

Libraries imported
Libraries imported


In [5]:
## 定义Sub-Agents的工具

In [6]:
from neo4j_for_adk import graphdb


def hello_world_tool(name: str) -> dict:
    """
    这是一个打招呼的工具。当用户要求你向某人说 Hello 或者打招呼时，请调用此工具。

    Args:
        name (str): 需要打招呼的人的名字。

    Returns:
        dict: 包含打招呼结果的字典。
    """
    # 这里是工具的核心逻辑（你可以把它想象成查数据库、调 API 的地方）
    greeting_message = f"Hello, {name}! 欢迎来到 Google ADK 的世界！"

    # 打印一条信息在控制台，方便我们肉眼观察工具是否真的被偷偷调用了
    print(f"\n[🔧 后台运行] hello_world_tool 被触发了，参数 name={name}")

    cypher_query = f"RETURN 'Hello to you, {name}' AS reply"

    return graphdb.send_query(cypher_query)

In [7]:
def say_goodbye_tool() -> dict:
    """
    这是一个道别/告别工具。当用户要求你向某人说再见、道别或结束对话时，请务必调用此工具。
    Returns:
        dict: 包含道别结果的字典。
    """


    # ==============================================================
    # 💡 进阶：如果你想和 hello 工具一样，通过 Neo4j 数据库来返回，
    # 可以把下面 return 字典的代码删掉，换成这句：
    # cypher_query = f"RETURN 'Goodbye to you, {name}' AS reply"
    # return graphdb.send_query(cypher_query)
    # ==============================================================

    # 默认返回给大模型的标准字典
    cypher_query = f"RETURN 'Goodbye to you' AS farewell"
    return graphdb.send_query(cypher_query)

In [8]:
## 定义Sub-Agents

In [14]:
from pathlib import Path

# 使用当前工作目录（Jupyter Notebook 中通常是启动 Notebook 的目录）
PROMPT_FILE = Path.cwd() / "greeting_subagent_instruction.md"
GREETING_SUBAGENT_INSTRUCTION = PROMPT_FILE.read_text(encoding="utf-8")

greeting_subagent = Agent(
    name="greeting_subagent_v1",
    description="这是一个专职的迎宾专员智能体。它的主要职责是使用专用的工具向新用户发送问候和打招呼。当有欢迎新客人的需求时，请呼叫此 Agent。",
    instruction=GREETING_SUBAGENT_INSTRUCTION,
    model=llm,
    tools=[hello_world_tool]
)

print(f"Agent '{greeting_subagent.name}' created")


Agent 'greeting_subagent_v1' created


In [15]:
# 告别subagent
# 1. 读取道别指令文件（假设文件名是 goodbye_subagent_prompt.md）
PROMPT_FILE = Path.cwd() / "goodbye_subagent_instruction.md"
GOODBYE_SUBAGENT_INSTRUCTION = PROMPT_FILE.read_text(encoding="utf-8")

# 2. 定义道别子代理
farewell_subagent = Agent(
    name="farewell_subagent_v1",
    description="这是一个专职的送别专员智能体。它的主要职责是向用户发送道别和再见。当需要结束对话或送别客人时，请呼叫此 Agent。",
    instruction=GOODBYE_SUBAGENT_INSTRUCTION,  # 直接填入从 .md 读取的字符串
    model=llm,
    tools=[say_goodbye_tool]
    # 如果不需要工具，可以省略 tools 参数；如果需要工具，请添加对应工具
)

print(f"Agent '{farewell_subagent.name}' created")

Agent 'farewell_subagent_v1' created


In [ ]:
# 创建根agent

In [16]:
root_agent = Agent(
    name="root_agent",
    description="总调度智能体，根据用户意图将任务委派给迎宾专员或送别专员。",
    instruction="""
你是一个总调度智能体，负责理解用户的意图，并将任务分配给合适的子智能体。

## 可用的子智能体
1. **GreetingAgent**：负责问候、欢迎、打招呼等场景。
2. **GoodbyeAgent**：负责道别、再见、结束对话等场景。

## 调度规则
- 如果用户表达的是问候、欢迎、打招呼等意图（例如说“你好”、“嗨”、“欢迎”），请调用 **GreetingAgent**。
- 如果用户表达的是道别、再见、结束对话等意图（例如说“再见”、“拜拜”、“我要走了”），请调用 **GoodbyeAgent**。
- 如果用户意图不明确，请礼貌地询问用户需要问候还是道别。
- 如果用户请求与问候或道别无关，请说明你只负责这些任务，并引导用户联系其他智能体。

## 输出要求
- 只调用相应的子智能体，不要自己生成问候语或道别语。
- 将子智能体返回的结果原样输出给用户。
""",
    model=llm,
    sub_agents=[greeting_subagent, farewell_subagent]  # 注意参数名是 sub_agents（复数）
)

print(f"Agent '{root_agent.name}' created")

Agent 'root_agent' created
